# 02 — Data Profiling & Validation

**Tujuan notebook ini (Fase 2 roadmap):**
- 2.1 Data Profiling — rows, columns, data types, missing values, duplicate keys per collection
- 2.2 Relationship Validation — cek foreign key (orphan records)
- 2.3 Granularity Check — pastikan grain tiap tabel dipahami dengan benar
- 2.4 Cek Dasar Repeat Purchase — checkpoint awal untuk target ML nanti (bukan keputusan final)

> Notebook ini masih di Fase A (cari fakta) sesuai prinsip "Roadmap = baseline, data aktual = hakimnya" — belum ada keputusan window/target yang dikunci di sini.


In [1]:
import pandas as pd
from pymongo import MongoClient

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "olist_db"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

print("Connected. Collections:", db.list_collection_names())


Connected. Collections: ['category_translation_raw', 'sellers_raw', 'geolocation_raw', 'orders_raw', 'payments_raw', 'products_raw', 'customers_raw', 'reviews_raw', 'order_items_raw']


### Load semua collection ke DataFrame

Sesuai Fase 0 #1 (raw data tidak langsung dimodifikasi) — ini cuma *read*, tidak ada write balik ke MongoDB di notebook ini.


In [2]:
COLLECTIONS = [
    "customers_raw", "orders_raw", "order_items_raw", "payments_raw",
    "reviews_raw", "products_raw", "sellers_raw", "geolocation_raw",
    "category_translation_raw",
]

dfs = {}
for col_name in COLLECTIONS:
    dfs[col_name] = pd.DataFrame(list(db[col_name].find({}, {"_id": 0})))
    print(f"{col_name}: {dfs[col_name].shape[0]} rows, {dfs[col_name].shape[1]} columns")


customers_raw: 99441 rows, 5 columns
orders_raw: 99441 rows, 8 columns
order_items_raw: 112650 rows, 7 columns
payments_raw: 103886 rows, 5 columns
reviews_raw: 99224 rows, 7 columns
products_raw: 32951 rows, 9 columns
sellers_raw: 3095 rows, 4 columns
geolocation_raw: 1000163 rows, 5 columns
category_translation_raw: 71 rows, 2 columns


---
## 2.1 Data Profiling

Untuk tiap collection: rows, columns, data types, missing values, unique values, duplicate keys.


In [3]:
def profile_collection(name, df, key_col=None):
    print(f"\n{'='*70}")
    print(f"COLLECTION: {name}")
    print(f"{'='*70}")
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nMissing values (kolom dengan missing > 0):")
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print(missing if len(missing) > 0 else "  (tidak ada missing value)")

    if key_col and key_col in df.columns:
        dup_count = df[key_col].duplicated().sum()
        print(f"\nDuplicate '{key_col}': {dup_count}")
        print(f"Unique '{key_col}': {df[key_col].nunique()}")

for name in COLLECTIONS:
    profile_collection(name, dfs[name])



COLLECTION: customers_raw
Shape: 99441 rows x 5 columns

Dtypes:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

Missing values (kolom dengan missing > 0):
  (tidak ada missing value)

COLLECTION: orders_raw
Shape: 99441 rows x 8 columns

Dtypes:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

Missing values (kolom dengan missing > 0):
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

COLLECTION: order_items_raw
Shape: 112650 rows x 7 columns

Dtypes:
order_id                   str
order_item_id            int64
pro

### Profiling khusus `orders_raw`

Sesuai Fase 2.1 — tambahan khusus: distribusi `order_status`, rentang tanggal pembelian, duplicate `order_id`.


In [4]:
orders = dfs["orders_raw"].copy()
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])

print("Distribusi order_status:")
print(orders["order_status"].value_counts())

print(f"\nRentang tanggal order_purchase_timestamp:")
print(f"  Min: {orders['order_purchase_timestamp'].min()}")
print(f"  Max: {orders['order_purchase_timestamp'].max()}")

print(f"\nDuplicate order_id: {orders['order_id'].duplicated().sum()}")
print(f"Unique order_id: {orders['order_id'].nunique()} (dari total {len(orders)} baris)")


Distribusi order_status:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Rentang tanggal order_purchase_timestamp:
  Min: 2016-09-04 21:15:19
  Max: 2018-10-17 17:30:18

Duplicate order_id: 0
Unique order_id: 99441 (dari total 99441 baris)


---
## 2.2 Relationship Validation

Cek foreign key — cari record yang tidak punya pasangan (orphan).


In [5]:
def check_orphans(child_df, child_key, parent_df, parent_key, label):
    child_keys = set(child_df[child_key].dropna())
    parent_keys = set(parent_df[parent_key].dropna())
    orphans = child_keys - parent_keys
    print(f"{label}: {len(orphans)} orphan value(s) dari {len(child_keys)} unique key di child")
    return orphans

# orders.customer_id -> customers.customer_id
orphan_orders_customers = check_orphans(
    dfs["orders_raw"], "customer_id", dfs["customers_raw"], "customer_id",
    "orders.customer_id -> customers.customer_id"
)

# order_items.order_id -> orders.order_id
orphan_items_orders = check_orphans(
    dfs["order_items_raw"], "order_id", dfs["orders_raw"], "order_id",
    "order_items.order_id -> orders.order_id"
)

# order_items.product_id -> products.product_id
orphan_items_products = check_orphans(
    dfs["order_items_raw"], "product_id", dfs["products_raw"], "product_id",
    "order_items.product_id -> products.product_id"
)

# order_items.seller_id -> sellers.seller_id
orphan_items_sellers = check_orphans(
    dfs["order_items_raw"], "seller_id", dfs["sellers_raw"], "seller_id",
    "order_items.seller_id -> sellers.seller_id"
)

# payments.order_id -> orders.order_id
orphan_payments_orders = check_orphans(
    dfs["payments_raw"], "order_id", dfs["orders_raw"], "order_id",
    "payments.order_id -> orders.order_id"
)

# reviews.order_id -> orders.order_id
orphan_reviews_orders = check_orphans(
    dfs["reviews_raw"], "order_id", dfs["orders_raw"], "order_id",
    "reviews.order_id -> orders.order_id"
)


orders.customer_id -> customers.customer_id: 0 orphan value(s) dari 99441 unique key di child
order_items.order_id -> orders.order_id: 0 orphan value(s) dari 98666 unique key di child
order_items.product_id -> products.product_id: 0 orphan value(s) dari 32951 unique key di child
order_items.seller_id -> sellers.seller_id: 0 orphan value(s) dari 3095 unique key di child
payments.order_id -> orders.order_id: 0 orphan value(s) dari 99440 unique key di child
reviews.order_id -> orders.order_id: 0 orphan value(s) dari 98673 unique key di child


---
## 2.3 Granularity Check (wajib)

Pastikan pemahaman grain tiap tabel benar sebelum lanjut ke fase manapun.


In [6]:
print("orders_raw:")
print(f"  1 row = 1 order? {dfs['orders_raw']['order_id'].is_unique}")

print("\norder_items_raw:")
print(f"  1 row = 1 produk dalam order (order_id BOLEH muncul berkali-kali): "
      f"{not dfs['order_items_raw']['order_id'].is_unique}")
print(f"  Rata-rata item per order: {len(dfs['order_items_raw']) / dfs['orders_raw']['order_id'].nunique():.2f}")

print("\ncustomers_raw:")
print(f"  1 row = 1 customer_id (unique)? {dfs['customers_raw']['customer_id'].is_unique}")
print(f"  Jumlah unique customer_id : {dfs['customers_raw']['customer_id'].nunique()}")
print(f"  Jumlah unique customer_unique_id: {dfs['customers_raw']['customer_unique_id'].nunique()}")
print(f"  -> customer_unique_id < customer_id berarti ada pelanggan dengan lebih dari 1 customer_id (lintas order)")

print("\ngeolocation_raw:")
print(f"  Jumlah baris: {len(dfs['geolocation_raw'])}")
print(f"  Unique zip_code_prefix: {dfs['geolocation_raw']['geolocation_zip_code_prefix'].nunique()}")
print(f"  -> confirm TIDAK 1:1 (baris jauh lebih banyak dari unique prefix), perlu agregasi nanti (Fase 3.4)")


orders_raw:
  1 row = 1 order? True

order_items_raw:
  1 row = 1 produk dalam order (order_id BOLEH muncul berkali-kali): True
  Rata-rata item per order: 1.13

customers_raw:
  1 row = 1 customer_id (unique)? True
  Jumlah unique customer_id : 99441
  Jumlah unique customer_unique_id: 96096
  -> customer_unique_id < customer_id berarti ada pelanggan dengan lebih dari 1 customer_id (lintas order)

geolocation_raw:
  Jumlah baris: 1000163
  Unique zip_code_prefix: 19015
  -> confirm TIDAK 1:1 (baris jauh lebih banyak dari unique prefix), perlu agregasi nanti (Fase 3.4)


---
## 2.4 Cek Dasar Repeat Purchase (checkpoint awal — BUKAN keputusan final)

> ⚠️ Ini cuma checkpoint awal. Keputusan final target ML baru diambil setelah observation/prediction window ditentukan dari distribusi tanggal aktual (Fase 7.1) dan positive case dicek ulang di dalam window tersebut (lihat bagian "Prinsip Pegangan Selama Implementasi" di roadmap).


In [7]:
customers = dfs["customers_raw"]
orders = dfs["orders_raw"].merge(
    customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left"
)

order_count_per_customer = orders.groupby("customer_unique_id")["order_id"].nunique()

total_unique_customers = order_count_per_customer.shape[0]
repeat_customers = (order_count_per_customer > 1).sum()
repeat_rate = repeat_customers / total_unique_customers * 100

print(f"Total unique customer_unique_id : {total_unique_customers}")
print(f"Customer dengan > 1 order        : {repeat_customers}")
print(f"Repeat purchase rate (keseluruhan dataset, TANPA window): {repeat_rate:.2f}%")

print(f"\nDistribusi jumlah order per customer:")
print(order_count_per_customer.value_counts().sort_index().head(10))


Total unique customer_unique_id : 96096
Customer dengan > 1 order        : 2997
Repeat purchase rate (keseluruhan dataset, TANPA window): 3.12%

Distribusi jumlah order per customer:
order_id
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64


**Interpretasi angka di atas:**
- Ini rate dari **seluruh histori dataset**, belum dipotong observation/prediction window — jadi ini bukan angka final untuk menentukan feasibility target ML.
- Kalau rate ini sudah terlihat sangat kecil (low single digit %), itu sinyal awal bahwa begitu dipotong ke window yang lebih pendek, positive case bisa jauh lebih sedikit lagi — perlu diperhitungkan serius di Fase 7.1 & Decision Gate.
- Langkah selanjutnya (bukan di notebook ini): plot distribusi `order_purchase_timestamp` untuk menentukan observation/prediction window, lalu hitung ulang repeat rate & jumlah positive case **di dalam window tersebut**.


---
## Definition of Done (Fase 2)

- [ ] Semua collection sudah diprofilling (rows, columns, missing values, duplicate keys)
- [ ] Relationship/foreign key sudah divalidasi (orphan records diketahui jumlahnya)
- [ ] Granularity tiap tabel dipahami dan dikonfirmasi lewat kode (bukan cuma asumsi)
- [ ] Repeat purchase rate dasar (tanpa window) sudah dihitung sebagai checkpoint awal

**Catatan:** kalau ada orphan record dalam jumlah signifikan di bagian 2.2, catat di `docs/data_dictionary.md` sebagai known data quality issue — akan ditangani di Fase 3 (Cleaning), bukan di notebook ini.

**Lanjut ke:** `03_data_cleaning_transformation.ipynb`
